# 1.6 — My Personal Chef

A **personal chef agent** that:

1. takes the **leftover ingredients** you have in your kitchen,
2. **searches the web** for recipes that use those ingredients,
3. returns the recipes to you,
4. **remembers previous runs**, so it can answer any follow-up question,
5. accepts a **photo of your fridge / ingredients** and tells you what you have and what you still need — *no text required*,
6. accepts **audio** questions (speak instead of type).

## Which techniques are used?

Everything below is built **only** with the techniques shown in notebooks **1.1 → 1.4**:

| Feature | Technique | Learned in |
|---|---|---|
| Chat model + agent | `init_chat_model`, `create_agent`, `system_prompt` | 1.1 |
| Clean ingredient list from a photo | Structured output (`response_format` + Pydantic) | 1.1 |
| Recipe lookup | `@tool` + Tavily `web_search` added to the agent | 1.2 |
| Memory of previous runs | `InMemorySaver` checkpointer + `thread_id` config | 1.3 |
| Photo input | Multimodal `HumanMessage` with a base64 **image** | 1.4 |
| Voice input | Multimodal `HumanMessage` with a base64 **audio** | 1.4 |

## Requirements

- `GOOGLE_API_KEY` (model) and `TAVILY_API_KEY` (web search) set in your `.env`
- packages from `requirements.txt` (`ipywidgets`, `sounddevice`, `scipy` for the photo/audio features)

Run the cells **top to bottom**, each feature lives in its own cell.

## 0 · Setup — load the environment and import everything

One single cell with all imports, so every following cell only contains **one function or one idea**.

In [ ]:
from dotenv import load_dotenv
from pydantic import BaseModel
from tavily import TavilyClient

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver

import base64
import io
import time

from ipywidgets import FileUpload
from IPython.display import display
import sounddevice as sd
from scipy.io.wavfile import write
from tqdm import tqdm

load_dotenv()

## 1 · The chat model (technique from 1.1)

We use Gemini through `init_chat_model`, exactly like in notebook 1.1.

In [ ]:
def get_model():
    """Initialise the chat model used by every part of the chef."""
    return init_chat_model(
        model="gemini-3.8-flash",
        model_provider="google_genai",
    )


model = get_model()

## 2 · Web search tool (technique from 1.2)

The chef needs to find recipes on the web. Just like in notebook 1.2 we create a
`TavilyClient` and wrap it in a `@tool`, which we will later attach to the agent.

In [ ]:
# One shared Tavily client (reads TAVILY_API_KEY from .env)
tavily_client = TavilyClient()

In [ ]:
@tool
def search_recipes(query: str) -> dict:
    """Search the web for recipes, cooking ideas or nutrition information.

    Use it with the ingredients the user has, e.g.
    "recipes with chicken, rice and leftover tomatoes".
    """
    return tavily_client.search(query, max_results=5)

In [ ]:
# Quick smoke test of the tool (same style as notebook 1.2)
result = search_recipes.invoke({"query": "quick recipes with eggs and leftover vegetables"})

for r in result.get("results", [])[:3]:
    print("-", r["title"], "|", r["url"])

## 3 · Structured output: a clean ingredient list (technique from 1.1)

For the **photo feature** we want a *reliable* list of ingredients (not a wall of text).
Notebook 1.1 showed `response_format=<Pydantic model>` — we use it here.

In [ ]:
class IngredientList(BaseModel):
    """Structured description of what is inside a fridge / on a table."""

    ingredients: list[str]      # what the model can see
    missing_staples: list[str]  # common staples that seem to be missing
    notes: str                  # condition / freshness, 1-2 sentences

In [ ]:
def build_ingredients_agent(model):
    """A small agent whose final answer is always a structured IngredientList (1.1)."""
    return create_agent(
        model=model,
        system_prompt=(
            "You are the vision assistant of a personal chef. "
            "The user sends a photo of their fridge or their ingredients. "
            "List every food ingredient you can actually see, then the staple items "
            "(oil, salt, onions, ...) that appear to be missing. Keep notes short."
        ),
        response_format=IngredientList,
    )


ingredients_agent = build_ingredients_agent(model)

## 4 · Memory (technique from 1.3)

`InMemorySaver` + a `thread_id` gives the chef a **conversation memory**:
every new run with the *same* thread sees the previous runs (follow-up questions,
ingredients you mentioned earlier, recipes already proposed).

In [ ]:
def get_checkpointer():
    """In-memory checkpointer -> the agent remembers previous runs (1.3)."""
    return InMemorySaver()


checkpointer = get_checkpointer()

# Every run started with this thread id shares the same memory
DEFAULT_THREAD_ID = "my-personal-chef"

## 5 · The Personal Chef agent

System prompt + agent builder: the agent gets the **chef persona**, the **web search
tool** and the **checkpointer** — everything from 1.1, 1.2 and 1.3 in one place.

In [ ]:
CHEF_SYSTEM_PROMPT = """You are My Personal Chef, a friendly home-cooking assistant.

Rules:
- The user tells you which leftover ingredients they have (or sends a photo of them).
- ALWAYS use the search_recipes tool to find real recipes on the web that use those
  ingredients before answering; never invent recipes from memory alone.
- Present 3 to 5 recipes: name, link, why it fits, and which of the user's
  ingredients it uses.
- Point out which ingredients are still missing (pantry staples count as missing).
- Remember the conversation: if the user asks a follow-up question
  ("step by step", "make it vegetarian", "what did I have again?"),
  answer from the history, do not start over.
- Be concise and practical."""


def build_chef_agent(model, checkpointer):
    """Create the Personal Chef agent: tools + system prompt + memory."""
    return create_agent(
        model=model,
        tools=[search_recipes],
        system_prompt=CHEF_SYSTEM_PROMPT,
        checkpointer=checkpointer,
    )


chef = build_chef_agent(model, checkpointer)

## 6 · Chatting with the chef — text, with memory (techniques from 1.1 + 1.3)

One function per cell: `ask_chef` sends a question and returns the answer.
Passing the `thread_id` config is what makes **previous runs** visible to the agent.

In [ ]:
def ask_chef(question: str, thread_id: str = DEFAULT_THREAD_ID) -> str:
    """Send a text question to the chef and get the answer back.

    The same thread_id => the chef remembers everything said before.
    """
    config = {"configurable": {"thread_id": thread_id}}

    response = chef.invoke(
        {"messages": [HumanMessage(content=question)]},
        config,
    )
    return response["messages"][-1].content

In [ ]:
# --- Run 1: tell the chef what is in the kitchen (first run) ---
answer = ask_chef("I have chicken thighs, rice, half an onion and two tomatoes. What can I cook?")
print(answer)

In [ ]:
# --- Run 2: follow-up question, same thread_id => the chef remembers run 1 ---
answer = ask_chef("Great, give me the full step-by-step instructions for the first one.")
print(answer)

In [ ]:
# --- Run 3: memory across runs -> it recalls what you told it earlier ---
answer = ask_chef("Which ingredients did I say I had at the very beginning?")
print(answer)

## 7 · Streaming the answer token by token (technique from 1.1)

Same call as above, but with `stream(..., stream_mode="messages")` so the recipe list
appears while it is being written.

In [ ]:
def stream_chef_answer(question: str, thread_id: str = DEFAULT_THREAD_ID):
    """Stream the chef's answer token by token (1.1 streaming)."""
    config = {"configurable": {"thread_id": thread_id}}

    for token, _metadata in chef.stream(
        {"messages": [HumanMessage(content=question)]},
        config,
        stream_mode="messages",
    ):
        if token.content:
            print(token.content, end="", flush=True)


stream_chef_answer("What can I cook with just pasta, garlic and olive oil?")

## 8 · Feature 1 — Photo of the fridge / ingredients (technique from 1.4)

**No typing needed**: upload a picture, the model reads it.

Step 1 — show the upload widget.

In [ ]:
def show_image_uploader(accept: str = ".jpg,.jpeg,.png") -> FileUpload:
    """Display a file-upload widget and return it (same as notebook 1.4)."""
    uploader = FileUpload(accept=accept, multiple=False)
    display(uploader)
    return uploader


uploader = show_image_uploader()

In [ ]:
def image_to_b64(uploader: FileUpload) -> str:
    """Convert the uploaded file into a base64 string (1.4: memoryview -> bytes -> b64)."""
    if not uploader.value:
        raise ValueError("No image uploaded yet - run the upload widget cell first.")

    uploaded_file = uploader.value[0]
    img_bytes = bytes(uploaded_file["content"])          # memoryview -> bytes
    return base64.b64encode(img_bytes).decode("utf-8")   # bytes -> base64 str


def image_mime_type(uploader: FileUpload) -> str:
    """Guess the mime type from the file name (image/jpeg, image/png)."""
    name = uploader.value[0]["name"].lower()
    if name.endswith(".png"):
        return "image/png"
    return "image/jpeg"

In [ ]:
def ingredients_from_photo(uploader: FileUpload) -> IngredientList:
    """Look at the photo and return a STRUCTURED ingredient list (1.4 image + 1.1 structured)."""
    img_b64 = image_to_b64(uploader)

    message = HumanMessage(content=[
        {"type": "text", "text": "Here is a photo of my fridge / my ingredients."},
        {"type": "image", "base64": img_b64, "mime_type": image_mime_type(uploader)},
    ])

    response = ingredients_agent.invoke({"messages": [message]})
    return response["structured_response"]

In [ ]:
def ask_chef_about_photo(
    uploader: FileUpload,
    question: str = "Suggest recipes I can cook with what you see in this photo.",
    thread_id: str = DEFAULT_THREAD_ID,
) -> str:
    """Send the PHOTO to the chef itself -> the photo becomes part of the memory,
    so later follow-up questions ("the second one, step by step") work too."""
    img_b64 = image_to_b64(uploader)

    message = HumanMessage(content=[
        {"type": "text", "text": question},
        {"type": "image", "base64": img_b64, "mime_type": image_mime_type(uploader)},
    ])

    response = chef.invoke(
        {"messages": [message]},
        {"configurable": {"thread_id": thread_id}},
    )
    return response["messages"][-1].content

In [ ]:
def fridge_workflow(uploader: FileUpload, thread_id: str = DEFAULT_THREAD_ID) -> str:
    """Complete photo workflow, no text needed:
    1. structured list of what is inside (what you have / what is missing)
    2. web-searched recipes using those ingredients
    """
    detected: IngredientList = ingredients_from_photo(uploader)
    have = ", ".join(detected.ingredients)

    print("🧴 Ingredients detected :", have)
    print("🧂 Missing staples      :", ", ".join(detected.missing_staples))
    print("📝 Notes                :", detected.notes)
    print("-" * 60)

    return ask_chef_about_photo(
        uploader,
        question=(
            "Here is a photo of my fridge. I can see: "
            f"{have}. Search the web for recipes using these ingredients "
            "and tell me which staples I am still missing."
        ),
        thread_id=thread_id,
    )

In [ ]:
# --- Demo: upload a photo in the widget cell above, then run this cell ---
if uploader.value:
    print(fridge_workflow(uploader))
else:
    print("Upload a photo first, then re-run this cell.")

## 9 · Feature 2 — Audio questions (technique from 1.4)

Speak instead of type: record with your microphone, encode as base64 WAV,
send as an **audio** part of a `HumanMessage`.

In [ ]:
def record_question(duration: int = 5, sample_rate: int = 44100) -> str:
    """Record `duration` seconds from the microphone and return base64 WAV (1.4 audio)."""
    print(f"Recording for {duration}s ...")
    audio = sd.rec(int(duration * sample_rate), samplerate=sample_rate, channels=1)

    for _ in tqdm(range(duration * 10)):   # progress bar, 10 updates per second
        time.sleep(0.1)
    sd.wait()
    print("Done.")

    buf = io.BytesIO()
    write(buf, sample_rate, audio)         # numpy array -> WAV bytes
    wav_bytes = buf.getvalue()

    return base64.b64encode(wav_bytes).decode("utf-8")

In [ ]:
def ask_chef_by_audio(
    aud_b64: str,
    instruction: str = "Listen to this question and answer it. "
                       "If the user mentions ingredients, search the web for recipes.",
    thread_id: str = DEFAULT_THREAD_ID,
) -> str:
    """Send an audio message to the chef (multimodal audio input, 1.4)."""
    message = HumanMessage(content=[
        {"type": "text", "text": instruction},
        {"type": "audio", "base64": aud_b64, "mime_type": "audio/wav"},
    ])

    response = chef.invoke(
        {"messages": [message]},
        {"configurable": {"thread_id": thread_id}},
    )
    return response["messages"][-1].content

In [ ]:
# --- Demo: record yourself asking a question, then the chef answers ---
question_audio = record_question(duration=5)
print(ask_chef_by_audio(question_audio))

## 10 · One entry point for everything

`personal_chef(...)` dispatches to text, photo or audio — always with the same memory
thread, so **all** modalities share one conversation.

In [ ]:
def personal_chef(
    text: str | None = None,
    uploader: FileUpload | None = None,
    aud_b64: str | None = None,
    thread_id: str = DEFAULT_THREAD_ID,
) -> str:
    """Unified entry point: give text, an uploaded photo and/or recorded audio."""
    if uploader is not None and uploader.value:
        question = text or "What can I cook with what you see in this photo?"
        return ask_chef_about_photo(uploader, question=question, thread_id=thread_id)

    if aud_b64 is not None:
        instruction = text or "Listen to this question and answer it."
        return ask_chef_by_audio(aud_b64, instruction=instruction, thread_id=thread_id)

    if text:
        return ask_chef(text, thread_id=thread_id)

    raise ValueError("Provide at least one of: text, uploader, aud_b64.")

In [ ]:
# --- Demo of the unified entry point ---
print(personal_chef(text="I have lentils, carrots and cumin - what can I make?"))

## 11 · Interactive session

A small chat loop for daily use. Commands:

- type anything → ask the chef (text, with memory)
- `/photo` → upload a fridge photo and get recipes
- `/audio` → record a spoken question
- `/quit` → leave

In [ ]:
def run_chef_session():
    """Interactive REPL for My Personal Chef."""
    print("🍳 My Personal Chef - /photo, /audio or /quit")
    while True:
        try:
            user_input = input("You: ").strip()
        except (EOFError, KeyboardInterrupt):
            break

        if not user_input:
            continue
        if user_input.lower() in {"/quit", "/exit", "quit", "exit"}:
            print("Bon appetit! 👋")
            break
        if user_input.lower() == "/photo":
            up = show_image_uploader()
            input("Upload the photo, then press Enter ...")
            try:
                print("\nChef:", fridge_workflow(up))
            except ValueError as e:
                print("Chef:", e)
            continue
        if user_input.lower() == "/audio":
            try:
                audio = record_question(duration=5)
                print("\nChef:", ask_chef_by_audio(audio))
            except Exception as e:
                print("Audio failed:", e)
            continue

        print("\nChef:", ask_chef(user_input))

In [ ]:
# Launch the interactive session (run it in a notebook cell)
# run_chef_session()

## Summary — what was built

| Cell / section | Function | Technique (notebook) |
|---|---|---|
| §1 | `get_model()` | `init_chat_model` (1.1) |
| §2 | `search_recipes` | `@tool` + Tavily web search (1.2) |
| §3 | `IngredientList`, `ingredients_agent` | Structured output (1.1) |
| §4 | `get_checkpointer()` | `InMemorySaver` memory (1.3) |
| §5 | `build_chef_agent()` | `create_agent` + system prompt (1.1) |
| §6 | `ask_chef()` | Agent invoke + `thread_id` memory (1.1 + 1.3) |
| §7 | `stream_chef_answer()` | Streaming (1.1) |
| §8 | `show_image_uploader()`, `image_to_b64()`, `ingredients_from_photo()`, `ask_chef_about_photo()`, `fridge_workflow()` | Image messages (1.4) + structured output (1.1) |
| §9 | `record_question()`, `ask_chef_by_audio()` | Audio messages (1.4) |
| §10 | `personal_chef()` | All of the above combined |
| §11 | `run_chef_session()` | Interactive loop |

**Ideas to go further:** save the checkpointer to disk so the memory survives restarts,
add a `@tool` that returns the full recipe page, or give the chef a second thread per recipe.